In [0]:
import requests
import json
from datetime import date, timedelta
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp, lit, col

## Adjusting Date Range for backfill. Then running daily and Pulling yesterday's date

In [0]:
# Backfill Widgets at top of screen
dbutils.widgets.text("start_date", "", "Backfill start date (YYYY-MM-DD, optional)")
dbutils.widgets.text("end_date", "", "Backfill end date (YYYY-MM-DD, optional)")

start_param = dbutils.widgets.get("start_date").strip()
end_param = dbutils.widgets.get("end_date").strip()

# If using the widgets, use those dates
if start_param and end_param:
    START_DATE = date.fromisoformat(start_param)
    END_DATE = date.fromisoformat(end_param)
    print(f"[BACKFILL MODE] {START_DATE} through {END_DATE}")
else:
    # Daily mode:
    # Pull yesterday for completed games,
    # plus today and the upcoming 45 days for the future schedule.
    START_DATE = date.today() - timedelta(days=1)
    END_DATE = date.today() + timedelta(days=45)

    print(f"[DAILY MODE] Pulling {START_DATE} through {END_DATE}")




# MLB Stats API Ingestion

In [0]:
base_url = "https://statsapi.mlb.com/api/v1"
bronze_schedule_table = "bronze.mlb_schedule_raw"
bronze_boxscore_table = "bronze.mlb_boxscore_raw"
bronze_teams_table = "bronze.mlb_teams_raw"

## Pulling daily schedule

In [0]:
def daterange(start: date, end: date):
    for n in range((end - start).days + 1):
        yield start + timedelta(days=n)

In [0]:
TERMINAL_STATUSES = {"Final", "Completed Early", "Postponed", "Cancelled", "Suspended"}
# Postponed/Cancelled are terminal for games.
# a makeup game gets its own separate game_pk on a new date, so we don't
# need to keep re-checking the original postponed entry.
 
def date_needs_refetch(game_date: date) -> bool:
    """True if this date was never fetched, or was fetched but still has
    a game in a non-terminal status (meaning something could still change).
    Scheduled games will have to be refetched."""
    try:
        rows = spark.table(bronze_schedule_table) \
            .filter(f"source_date = '{game_date.isoformat()}'") \
            .orderBy(col("ingestion_timestamp").desc()) \
            .select("raw_json").limit(1).collect()
    except Exception:
        return True  # table doesn't exist yet — first run
 
    if not rows:
        return True  # never fetched
 
    parsed = json.loads(rows[0].raw_json)
    for day in parsed.get("dates", []):
        for game in day.get("games", []):
            status = game.get("status", {}).get("detailedState")
            if status not in TERMINAL_STATUSES:
                return True  # still pending — needs a fresh pull
    return False

In [0]:
dates_to_fetch = [d for d in daterange(START_DATE, END_DATE) if date_needs_refetch(d)]
dates_skipped = (END_DATE - START_DATE).days + 1 - len(dates_to_fetch)
 
print(f"Dates needing fetch: {len(dates_to_fetch)}")
print(f"Dates skipped (already fully terminal): {dates_skipped}")

## Fetching the MLB schedule for a single date and returning the raw JSON

In [0]:
def fetch_schedule(game_date: date) -> dict:
    """Fetch the MLB schedule for a single date. Returns raw JSON."""
    url = f"{base_url}/schedule"
    params = {
        "sportId": 1,   # 1 = MLB
        "date": game_date.strftime("%m/%d/%Y")
    }
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    return resp.json()

In [0]:
schedule_rows = []

for d in dates_to_fetch:
    try:
        raw = fetch_schedule(d)
    except requests.RequestException as e:
        # log error and continue
        print(f"[WARN] schedule fetch failed for {d}: {e}")
        continue

    schedule_rows.append(
        Row(
            source_date = d.isoformat(),
            raw_json = json.dumps(raw)
        )
    )

print(f"Fetched schedule for {len(schedule_rows)} dates")

## Write schedule to Bronze

In [0]:
if schedule_rows:

    schedule_df = spark.createDataFrame(schedule_rows) \
        .withColumn("ingestion_timestamp", current_timestamp())

    # Make the newly fetched schedules available to SQL
    schedule_df.createOrReplaceTempView("new_schedule")

    # Remove the previous snapshot for each date being refreshed
    spark.sql(f"""
        DELETE FROM {bronze_schedule_table}
        WHERE source_date IN (
            SELECT source_date
            FROM new_schedule
        )
    """)

    # Insert the newest snapshot
    schedule_df.write.format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(bronze_schedule_table)

    display(schedule_df)

## Extract gamePks from schedule JSON

In [0]:
def extract_game_pks(schedule_json: dict) -> list[int]:
    """Pull gamePk values out of a schedule reponse, skipping games that have no gamePk (shouldn't happen, but defensive)."""
    pks = []
    for day in schedule_json.get("dates", []):
        for game in day.get("games", []):
            pk = game.get("gamePk")
            if pk is not None:
                pks.append(pk)

    return pks

In [0]:
all_game_pks = []
for row in schedule_rows:
    raw = json.loads(row.raw_json)
    all_game_pks.extend(extract_game_pks(raw))

all_game_pks = sorted(set(all_game_pks))   # dedupe in case of overlappiing pulls
print(f"Found {len(all_game_pks)} unique games needing a boxscore pull")

## Pulling boxscores for each game

In [0]:
import time

def fetch_boxscore(game_pk: int) -> dict:
    url = f"{base_url}/game/{game_pk}/boxscore"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    return resp.json()

In [0]:
boxscore_rows = []
failed_pks = []

for pk in all_game_pks:
    try:
        raw = fetch_boxscore(pk)
        boxscore_rows.append(
            Row(
                game_pk = pk,
                raw_json = json.dumps(raw)
            )
        )
    except requests.RequestException as e:
        print(f"[WARN] boxscore fetch failed for gamePk {pk}: {e}")
        failed_pks.append(pk)
    time.sleep(0.1)   # small politeness delay

print(f"Fetched {len(boxscore_rows)} boxscores, {len(failed_pks)} failed")

## Writing boxscores to Bronze

In [0]:
if boxscore_rows:
    boxscore_df = spark.createDataFrame(boxscore_rows) \
        .withColumn("ingestion_timestamp", current_timestamp())

    boxscore_df.write.format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(bronze_boxscore_table)

    display(boxscore_df)

## Pulling Team reference data

In [0]:
def fetch_teams() -> dict:
    url = f"{base_url}/teams"
    resp = requests.get(url, params={"sportId": 1}, timeout=30)
    resp.raise_for_status()
    return resp.json()

In [0]:
try:
    teams_raw = fetch_teams()
    teams_row = [Row(raw_json=json.dumps(teams_raw))]

    teams_df = spark.createDataFrame(teams_row) \
        .withColumn("ingestion_timestamp",current_timestamp())

    teams_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(bronze_teams_table)

    print("Teams reference data landed in Bronze.")
    display(teams_df)
except requests.RequestException as e:
    print(f"[WARN] teams fetch failed: {e}")


## Writing Teams into Bronze

In [0]:
print("Schedule dates:", spark.table("bronze.mlb_schedule_raw").count())
print("Boxscore rows:", spark.table("bronze.mlb_boxscore_raw").count())
print("Distinct game_pks:", spark.table("bronze.mlb_boxscore_raw").select("game_pk").distinct().count())
print("Teams rows:", spark.table("bronze.mlb_teams_raw").count())

In [0]:
%sql
SELECT
    source_date,
    COUNT(*) AS rows_for_date,
    COUNT(DISTINCT raw_json) AS unique_jsons
FROM bronze_schedule_table
GROUP BY source_date
ORDER BY source_date;